# Quasi-norm Muon: run Hypothesis 1 on CIFAR-10 (Colab)

ViT (dim 256, depth 6, ~3.2M params) on CIFAR-10: Muon vs. Kaon vs. Freon (c = 2/3, 3/4) vs. SGD / TruncatedSGD.

**Runtime → Change runtime type → GPU** (A100 if you can get it; T4 works but is much slower).
Run top to bottom. Every sweep is **resumable**: if Colab disconnects, re-run the setup cells and the
sweep cell again and it skips finished runs (results are saved to Drive if you mount it).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

## 1. Setup (results go to Drive if mounted)

In [ ]:
import os
USE_DRIVE = True   # set False to keep results only on the (ephemeral) Colab disk
RESULTS = "/content/results"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS = "/content/drive/MyDrive/cs182_results"
os.makedirs(RESULTS, exist_ok=True)
print("results dir:", RESULTS)

In [ ]:
%cd /content
!git clone https://github.com/BistroP/CS182-final-project.git repo 2>/dev/null || (cd repo && git pull)
%cd /content/repo
!python -m pytest -q tests

In [ ]:
# Downloads CIFAR-10 (~170 MB) to data/cifar10 via torchvision, then a 1-epoch smoke run on it
!python train_cifar.py --optimizer muon --epochs 1 --out_dir runs/cifar/smoke

## 2. Benchmark (≈ 3–5 min) — do this first
Times one CIFAR epoch (97 steps at batch 512, + test eval) per optimizer. Look at the **freon svd vs eigh**
rows and pick the faster one for `FREON_IMPL` below. The last column extrapolates the whole stage-1 + stage-2 cost.

In [ ]:
import time
from train_cifar import CifarConfig, run

BENCH = {"muon": {}, "kaon": {}, "freon svd": dict(optimizer="freon", freon_impl="svd"),
         "freon eigh": dict(optimizer="freon", freon_impl="eigh"), "sgd": dict(momentum=0.9),
         "truncsgd": dict(momentum=0.9)}
EPOCHS_PER_OPT = 16 * 30 + 3 * 30   # stage 1 (16-point grid x 30 epochs) + stage 2 (3 seeds x 30 epochs)
sec_per_epoch = {}
for name, kw in BENCH.items():
    kw = {"optimizer": name.split()[0], **kw}
    res = run(CifarConfig(**kw, epochs=2, out_dir=f"runs/cifar/bench_{name.replace(' ', '_')}"), verbose=False)
    sec_per_epoch[name] = res["log"][-1]["time"] - res["log"][-2]["time"]   # last epoch only (skips warm-up)
    print(f"{name:12s} {sec_per_epoch[name]:6.2f} s/epoch   ~{sec_per_epoch[name] * EPOCHS_PER_OPT / 3600:5.1f} h for H1")

In [ ]:
FREON_IMPL = "eigh"   # <- set to "svd" if the benchmark shows svd is faster

## 3. Stage 1 — tune learning rates (seed 42, 30 epochs per run)
16 `(matrix_lr, base_lr)` points per spectral optimizer (15 for SGD / TruncatedSGD, which have their own larger-lr grid).
That's 16×4 + 15×2 = 94 runs. Kaon is Muon-cost; Freon cost depends on the benchmark.

To save compute you can tune on shorter runs (e.g. `TUNE_EPOCHS = 15`), at the cost that the best lr can shift a
little with run length — same caveat as the Shakespeare protocol.

In [ ]:
OPTS = "muon kaon freon:0.6667 freon:0.75 sgd truncsgd:0.05"
TUNE_EPOCHS = 30
!python sweep_cifar.py tune --opts {OPTS} --freon_impl {FREON_IMPL} --set epochs={TUNE_EPOCHS} --out {RESULTS}/cifar_tune.jsonl

In [ ]:
# Best spec:matrix_lr:base_lr per optimizer (by final test acc); warns if the best lr is on the grid edge
!python analyze_cifar.py tune {RESULTS}/cifar_tune.jsonl

## 4. Stage 2 — final runs, 3 seeds × 30 epochs at each optimizer's best lrs
Paste the `spec:matrix_lr:base_lr` strings from the `scale command` printed above into `FINAL_SPECS`.
(`--rhos 1` = no rescaling. For the paper's Figure-2 learning-rate sensitivity, run the optional cell in section 6.)

In [ ]:
FINAL_SPECS = "muon:0.02:0.001 kaon:0.02:0.001 freon:0.6667:0.02:0.001 freon:0.75:0.02:0.001 sgd:0.2:0.001 truncsgd:0.05:0.2:0.001"  # <- REPLACE with the best specs from stage 1
!python sweep_cifar.py scale --opts {FINAL_SPECS} --rhos 1 --seeds 42 43 44 --freon_impl {FREON_IMPL} --out {RESULTS}/cifar_final.jsonl

## 5. Summary: mean ± std final test accuracy over seeds

In [ ]:
import json, collections, statistics as st
rows = [json.loads(l) for l in open(f"{RESULTS}/cifar_final.jsonl") if l.strip()]
by = collections.defaultdict(list)
for r in rows:
    by[r["tag"]].append(r)
print(f"{'optimizer':16s} {'test acc':>17s} {'test loss':>17s}  n  diverged")
for tag, rs in sorted(by.items(), key=lambda kv: -st.mean(r["final_test_acc"] for r in kv[1])):
    acc = [r["final_test_acc"] for r in rs]; loss = [r["final_test_loss"] for r in rs]
    sd = lambda v: st.stdev(v) if len(v) > 1 else float("nan")
    print(f"{tag:16s} {st.mean(acc):.4f} ± {sd(acc):.4f}  {st.mean(loss):.4f} ± {sd(loss):.4f}  {len(rs)}  "
          f"{sum(r['diverged'] for r in rs)}")

## 6. (Optional, if time) learning-rate sensitivity — paper Fig. 2
Scales each tuned pair by ρ ∈ {0.03, 0.1, 0.3, 1, 3, 10, 30} over 3 seeds (7× the stage-2 cost).
ρ = 1 rows are recomputed, not reused, since they go to a separate file.

In [ ]:
!python sweep_cifar.py scale --opts {FINAL_SPECS} --freon_impl {FREON_IMPL} --out {RESULTS}/cifar_scale.jsonl
!python analyze_cifar.py scale {RESULTS}/cifar_scale.jsonl
from IPython.display import Image; Image(f"{RESULTS}/cifar_scale.png")